## 4 — Gráficos: Visão Por Indicador

| Gráfico | Tipo | Descrição |
|--------|------|-----------|
| 1  | Barras agrupadas | Indicador selecionado por tipo de curso e ano |
| 2 | Barras horizontais | Ranking do indicador por curso (último ano) |
| 3  | Heatmap | Indicador por Curso × Ano |
| 4  | Heatmap | Todos os indicadores por Curso (último ano) |
| 5 | Boxplot | Distribuição do indicador por tipo de curso (último ano) |
| 6 | Radar | Comparação de TC, TE, TR, IEf, TPE e TEFAcad por tipo de curso (último ano) |

### 4.1. Importações e configurações

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

CORES_INDICADORES = {
    "TC":      "#2196F3",
    "TE":      "#F44336",
    "TR":      "#FF9800",
    "IEf":     "#4CAF50",
    "TPE":     "#9C27B0",
    "TEFAcad": "#00BCD4",
}

# Indicador a analisar nos gráficos:
# Trocar para "TE", "TR", "IEf", "TPE" ou "TEFAcad" conforme necessário.
INDICADOR = "TC"

print("Indicador selecionado:", INDICADOR)

### 4.2. Carga dos dados e cálculo dos indicadores

In [ ]:
df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Shape:", df_dados_tratados.shape)
df_dados_tratados.head(3)

In [ ]:
df_dados_tratados.info()

In [ ]:
def calcular_indicadores(df, agrupamento):
    
    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )

    df_indicadores["t_matr_cont_reg"] = df_indicadores["MREG"] / ma * 100
    df_indicadores["TPE"]             = df_indicadores["TC"] + df_indicadores["t_matr_cont_reg"]
    df_indicadores["TEFAcad"]         = (
        df_indicadores["conc_no_prazo"]
        / df_indicadores["conc_previstos"].replace(0, np.nan) * 100
    )

    return df_indicadores.fillna(0).round(2)


# Calcula para diferentes granularidades
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])
ind_ano_tipo  = calcular_indicadores(df_dados_tratados, ["Ano", "Tipo de Curso"])

# Identifica o último ano disponível (usado nos gráficos 2, 5 e 6)
ultimo_ano = int(ind_ano_curso["Ano"].max())

print("ind_ano_curso:", ind_ano_curso.shape)
print("Último ano disponível:", ultimo_ano)

### 4.3. Gráficos

In [ ]:
# 6: Indicador por tipo de curso e ano

fig_g6 = px.bar(
    ind_ano_tipo,
    x="Ano",
    y=INDICADOR,
    color="Tipo de Curso",
    barmode="group",
    title=f"G6 — {INDICADOR} por Tipo de Curso e Ano",
    labels={INDICADOR: f"{INDICADOR} (%)", "Tipo de Curso": "Tipo"},
    text_auto=".1f",
)
fig_g6.update_xaxes(tickmode="linear", dtick=1)
fig_g6.show()

In [ ]:
# 2 : Ranking do indicador por curso — APENAS O ÚLTIMO ANO

# Filtra somente os dados do último ano
ind_ultimo_ano_curso = ind_ano_curso[ind_ano_curso["Ano"] == ultimo_ano]

# Ordena do menor para o maior para que a barra maior fique no topo
ranking = ind_ultimo_ano_curso[["Nome de Curso", INDICADOR]].sort_values(
    INDICADOR, ascending=True
)

# Define a escala de cor: vermelho para indicadores negativos, azul para positivos
escala = "Reds" if INDICADOR in ["TE", "TR"] else "Blues"

fig_g7 = px.bar(
    ranking,
    x=INDICADOR,
    y="Nome de Curso",
    orientation="h",
    color=INDICADOR,
    color_continuous_scale=escala,
    text_auto=".1f",
    title=f"G7 — Ranking de {INDICADOR} por Curso ({ultimo_ano})",
    labels={INDICADOR: f"{INDICADOR} (%)", "Nome de Curso": ""},
)
fig_g7.update_layout(coloraxis_showscale=False)
fig_g7.show()

In [ ]:
# 3: Heatmap do indicador por Curso × Ano
# Cada célula = valor do indicador para um curso em um ano.
# pivot_table() reorganiza o DataFrame: cursos nas linhas, anos nas colunas.

pivot = ind_ano_curso.pivot_table(
    index="Nome de Curso",
    columns="Ano",
    values=INDICADOR,
    fill_value=0,
)

escala_heatmap = "Reds" if INDICADOR in ["TE", "TR"] else "Blues"

fig_g8 = px.imshow(
    pivot,
    text_auto=".1f",
    color_continuous_scale=escala_heatmap,
    labels={"color": f"{INDICADOR} (%)"},
    title=f"G8 — Mapa de Calor: {INDICADOR} por Curso e Ano",
    aspect="auto",
)
fig_g8.show()

In [ ]:
# 4: Heatmap de todos os indicadores por curso (último ano)

todos_indicadores = ["TC", "TE", "TR", "IEf", "TPE", "TEFAcad"]

# Seleciona apenas o último ano e usa o curso como índice
pivot_todos = (
    ind_ano_curso[ind_ano_curso["Ano"] == ultimo_ano]
    .set_index("Nome de Curso")[todos_indicadores]
)

fig_g9 = px.imshow(
    pivot_todos,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    labels={"color": "(%)"},
    title=f"G9 — Todos os Indicadores por Curso ({ultimo_ano})",
    aspect="auto",
)
fig_g9.show()

In [ ]:
# 5: Boxplot do indicador por tipo de curso — APENAS O ÚLTIMO ANO
# points="all" mostra todos os cursos individualmente.

ind_ultimo_ano_tipo = ind_ano_tipo[ind_ano_tipo["Ano"] == ultimo_ano]

fig_g10 = px.box(
    ind_ultimo_ano_tipo,
    x="Tipo de Curso",
    y=INDICADOR,
    color="Tipo de Curso",
    points="all",
    title=f"G10 — Distribuição de {INDICADOR} por Tipo de Curso ({ultimo_ano})",
    labels={INDICADOR: f"{INDICADOR} (%)", "Tipo de Curso": ""},
)
fig_g10.update_layout(showlegend=False, xaxis_tickangle=-20)
fig_g10.show()

In [ ]:
# 6: Radar de indicadores por tipo de curso — APENAS O ÚLTIMO ANO 
# O polígono precisa ser "fechado" repetindo o primeiro valor ao final.

categorias_radar = ["TC", "TE", "TR", "IEf", "TPE", "TEFAcad"]

df_radar = ind_ano_tipo[ind_ano_tipo["Ano"] == ultimo_ano]

fig_g11 = go.Figure()

for _, linha in df_radar.iterrows():
    valores = [linha[c] for c in categorias_radar]
    # Repete o primeiro valor para fechar o polígono
    valores_fechados = valores + [valores[0]]
    cats_fechadas    = categorias_radar + [categorias_radar[0]]

    fig_g11.add_trace(go.Scatterpolar(
        r=valores_fechados,
        theta=cats_fechadas,
        fill="toself",
        name=linha["Tipo de Curso"],
    ))

fig_g11.update_layout(
    title=f"G11 — Radar de Indicadores por Tipo de Curso ({ultimo_ano})",
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    legend=dict(font_size=10),
)
fig_g11.show()